# 12 — Computation Graphs and Backpropagation

Backpropagation is an efficient way to apply the chain rule through a computation graph. A forward pass computes values from inputs to output; a backward pass starts at the final scalar loss and propagates derivatives in the reverse direction.

This notebook begins with scalar arithmetic, where every derivative can be checked by hand. It then introduces branches, vector operations, gradient shapes, ReLU, and numerical gradient checking.

## Learning objectives

- represent an expression as a computation graph;
- distinguish forward values, local derivatives, and upstream gradients;
- apply the chain rule one operation at a time;
- add gradient contributions when one value feeds multiple branches;
- reason about gradients of vector and matrix operations by shape;
- understand the ReLU derivative and its behavior at zero;
- compare analytical derivatives with centered numerical differences;
- backpropagate through a small affine–ReLU computation.


In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


## 1. A computation graph

Consider

$$
q = x + y,
\qquad
f = qz.
$$

The graph contains two simple operations:

```text
x ──┐
    +── q ──┐
y ──┘       ×── f
        z ──┘
```

For $x=-2$, $y=5$, and $z=-4$, the forward pass moves left to right:

$$
q = -2 + 5 = 3,
\qquad
f = 3(-4) = -12.
$$

The backward pass moves right to left. Because $f$ is the final scalar output, it begins with

$$
\frac{\partial f}{\partial f}=1.
$$

For multiplication $f=qz$, the local derivatives are

$$
\frac{\partial f}{\partial q}=z,
\qquad
\frac{\partial f}{\partial z}=q.
$$

For addition $q=x+y$,

$$
\frac{\partial q}{\partial x}=1,
\qquad
\frac{\partial q}{\partial y}=1.
$$

The chain rule connects the two operations:

$$
\frac{\partial f}{\partial x}
=
\frac{\partial f}{\partial q}
\frac{\partial q}{\partial x}.
$$

The same reasoning gives $\partial f/\partial y$.


### Exercise 1 — Complete a scalar forward and backward pass

Compute the forward values and all three input gradients. Write the equations by hand before filling the code.


In [45]:
x = -2.0
y = 5.0
z = -4.0

# Forward pass
q = x + y
f = q * z

# Backward pass: start with df/df = 1.
df = 1.0
dq = df * z  # df * local derivative of f with respect to q
dz = df * q  # df * local derivative of f with respect to z
dx = dq * 1  # dq * local derivative of q with respect to x
dy = dq * 1  # dq * local derivative of q with respect to y

assert f == -12.0
np.testing.assert_allclose([dx, dy, dz], [-4.0, -4.0, 3.0])
print(f"q={q}, f={f}")
print(f"dx={dx}, dy={dy}, dz={dz}")


q=3.0, f=-12.0
dx=-4.0, dy=-4.0, dz=3.0


## 2. Local and upstream gradients

Suppose a node computes

$$
u = g(a,b)
$$

and later operations produce a scalar loss $L$. During the backward pass the node receives the **upstream gradient**

$$
\frac{\partial L}{\partial u}.
$$

The node already knows its **local derivatives**

$$
\frac{\partial u}{\partial a}
\quad\text{and}\quad
\frac{\partial u}{\partial b}.
$$

It returns

$$
\frac{\partial L}{\partial a}
=
\frac{\partial L}{\partial u}
\frac{\partial u}{\partial a},
\qquad
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial u}
\frac{\partial u}{\partial b}.
$$

A useful phrase is:

> downstream gradient = upstream gradient × local gradient

“Upstream” and “downstream” refer to the **backward pass**. Although values flow toward the loss in the forward pass, gradients flow away from it in reverse.

### Common scalar gates

For $u=a+b$, both local derivatives are $1$, so addition copies the upstream gradient.

For $u=ab$,

$$
\frac{\partial u}{\partial a}=b,
\qquad
\frac{\partial u}{\partial b}=a,
$$

so multiplication swaps the forward inputs and scales the upstream gradient.

For $u=\max(0,a)$, the local derivative is $1$ when $a>0$ and $0$ when $a<0$.


## 3. Branches require gradient accumulation

Now consider

$$
a=x+y,
\qquad
b=x+z,
\qquad
f=ab.
$$

The value $x$ influences $f$ through **two paths**:

```text
          ┌── a ──┐
x ────────┤       ×── f
          └── b ──┘
```

The total derivative must include both contributions:

$$
\frac{\partial f}{\partial x}
=
\frac{\partial f}{\partial a}
\frac{\partial a}{\partial x}
+
\frac{\partial f}{\partial b}
\frac{\partial b}{\partial x}.
$$

This is why backward implementations use `+=` when multiple later operations send gradients into the same value. The chain rule multiplies derivatives **along** one path; branches are combined by adding **across** paths.


### Exercise 2 — Backpropagate through a branch

For $x=2$, $y=3$, and $z=-1$, calculate the forward output and the gradients with respect to all inputs. Make the two contributions to `dx` explicit.


In [46]:
x = 2.0
y = 3.0
z = -1.0

# Forward pass
a = x + y
b = x + z
f = a*b

# Backward through f = a * b.
df = 1.0
da = df * b
db = df * a

# Backward through a = x + y and b = x + z.
dx_from_a = da * 1
dx_from_b = db * 1
dx = dx_from_a + dx_from_b  # sum both paths
dy = da * 1
dz = db * 1

assert f == 5.0
np.testing.assert_allclose([dx, dy, dz], [6.0, 1.0, 5.0])
print(f"f={f}, dx={dx}, dy={dy}, dz={dz}")


f=5.0, dx=6.0, dy=1.0, dz=5.0


## 4. Backward functions consume an upstream gradient

A local operation should not assume that its output is the final loss. Its backward function therefore accepts `dout`, representing the derivative of the eventual loss with respect to this operation's output.

For multiplication $out=xy$:

$$
dx = dout \cdot y,
\qquad
dy = dout \cdot x.
$$

For addition $out=x+y$:

$$
dx=dout,
\qquad
dy=dout.
$$

The forward pass can cache inputs needed later. This forward/backward interface will become the basis of reusable neural-network layers.


### Exercise 3 — Implement scalar operation pairs

Implement both backward functions and verify that an arbitrary upstream gradient is applied.


In [6]:
def add_forward(x: float, y: float) -> tuple[float, tuple[float, float]]:
    out = x + y
    cache = (x, y)
    return out, cache


def add_backward(dout: float, cache: tuple[float, float]) -> tuple[float, float]:
    # Return derivatives of the final loss with respect to x and y.
    return (dout * 1, dout * 1)


def multiply_forward(x: float, y: float) -> tuple[float, tuple[float, float]]:
    out = x * y
    cache = (x, y)
    return out, cache


def multiply_backward(
    dout: float, cache: tuple[float, float]
) -> tuple[float, float]:
    return (dout * cache[1], dout * cache[0])


product, product_cache = multiply_forward(3.0, -2.0)
dx, dy = multiply_backward(4.0, product_cache)

assert product == -6.0
np.testing.assert_allclose([dx, dy], [-8.0, 12.0])
print(product, dx, dy)


-6.0 -8.0 12.0


## 5. From scalars to vectors and matrices

For one example, an affine operation is

$$
s = xW+b,
$$

with shapes

```text
x:     (D,)
W:     (D, C)
b:     (C,)
s:     (C,)
```

Assume a later computation sends an upstream gradient $ds$ with shape $(C,)$. The backward equations are

$$
dx = dsW^T,
$$

$$
dW = x^T ds,
$$

$$
db = ds.
$$

Because $x$ and $ds$ are one-dimensional NumPy arrays, the outer product is written as

```python
dW = np.outer(x, ds)
```

Shape reasoning is a powerful check:

```text
dx:  (C,) @ (C, D) -> (D,)
dW:  outer((D,), (C,)) -> (D, C)
db:  (C,)
```

The upstream gradient is essential. If the scalar objective is

$$
L = \sum_c s_c u_c,
$$

then $ds=u$. Backpropagation answers how $L$ changes, not merely how the intermediate scores change.


### Exercise 4 — Backpropagate through one affine example

Implement the forward and backward functions. The cache should contain the values required by the backward pass.


In [42]:
def affine_forward_single(
    x: np.ndarray, W: np.ndarray, b: np.ndarray
) -> tuple[np.ndarray, tuple[np.ndarray, np.ndarray]]:
    out = x @ W + b
    cache = (x, W)
    return (out, cache)


def affine_backward_single(
    ds: np.ndarray, cache: tuple[np.ndarray, np.ndarray]
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    W_T = cache[1].T
    x = cache[0]
    return (
        ds @ W_T,           # dx
        np.outer(x, ds),    # dW
        ds,                 # db
    )


x = np.array([1.0, -2.0])
W = np.array([[2.0, -1.0, 3.0], [0.5, 4.0, -2.0]])
b = np.array([0.1, -0.2, 0.3])
upstream = np.array([1.0, -2.0, 0.5])

scores, cache = affine_forward_single(x, W, b)
dx, dW, db = affine_backward_single(upstream, cache)

np.testing.assert_allclose(scores, [1.1, -9.2, 7.3])
np.testing.assert_allclose(dx, [5.5, -8.5])
np.testing.assert_allclose(dW, np.outer(x, upstream))
np.testing.assert_allclose(db, upstream)
assert dx.shape == x.shape
assert dW.shape == W.shape
assert db.shape == b.shape
print("scores:", scores)
print("dx:", dx)
print("dW:\n", dW)
print("db:", db)


scores: [ 1.1 -9.2  7.3]
dx: [ 5.5 -8.5]
dW:
 [[ 1.  -2.   0.5]
 [-2.   4.  -1. ]]
db: [ 1.  -2.   0.5]


## 6. ReLU and nondifferentiable points

The rectified linear unit is

$$
\operatorname{ReLU}(x)=\max(0,x).
$$

Away from zero its derivative is

$$
\frac{d}{dx}\operatorname{ReLU}(x)
=
\begin{cases}
0, & x<0,\\
1, & x>0.
\end{cases}
$$

At exactly $x=0$, the left derivative is $0$ and the right derivative is $1$, so the ordinary derivative does not exist. Implementations choose a convention, commonly gradient $0$ at zero.

For an array, the mask is elementwise:

$$
dx = dout \odot \mathbb{1}[x>0],
$$

where $\odot$ means elementwise multiplication.


### Exercise 5 — Implement the ReLU backward pass

Preserve the upstream gradient at positive inputs and block it at zero and negative inputs. Do not modify `dout` in place.


In [47]:
def relu_forward(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    out = np.maximum(0.0, x)
    cache = x
    return (out, cache)


def relu_backward(dout: np.ndarray, cache: np.ndarray) -> np.ndarray:
    return dout * (cache > 0)
    # list comprehension only handles one-dimensional arrays
    # return np.array([
    #     dout[i] if x > 0 else 0.0 for i, x in enumerate(cache)
    # ])


x = np.array([-2.0, 0.0, 3.0, 1.5])
upstream = np.array([10.0, 20.0, 30.0, -4.0])

out, cache = relu_forward(x)
dx = relu_backward(upstream, cache)

np.testing.assert_allclose(out, [0.0, 0.0, 3.0, 1.5])
np.testing.assert_allclose(dx, [0.0, 0.0, 30.0, -4.0])
np.testing.assert_allclose(upstream, [10.0, 20.0, 30.0, -4.0])
print("out:", out)
print("dx:", dx)


out: [0.  0.  3.  1.5]
dx: [ 0.  0. 30. -4.]


## 7. Numerical gradient checking

For a scalar function $f(x)$, the centered-difference approximation is

$$
f'(x)
\approx
\frac{f(x+h)-f(x-h)}{2h}.
$$

It is usually more accurate than the one-sided approximation because leading truncation errors cancel.

The step $h$ must be small, but not arbitrarily small:

- if $h$ is too large, the approximation does not measure sufficiently local behavior;
- if $h$ is too small, floating-point subtraction loses precision.

Values near $10^{-5}$ are useful starting points for many checks with `float64`, but the best value depends on the function and numerical scale.

Numerical gradients are slow because every input coordinate requires additional forward evaluations. They are debugging tools, not training algorithms.


### Exercise 6 — Implement a centered numerical gradient

The function should return one partial derivative for every element of `x`. Modify one coordinate at a time, restore it afterward, and avoid replacing the input array.


In [41]:
def eval_numerical_gradient(
    f,
    x: np.ndarray,
    h: float = 1e-5,
) -> np.ndarray:
    """Estimate the gradient of a scalar-valued function at x."""
    gradient = np.zeros_like(x, dtype=np.float64)

    for index in np.ndindex(x.shape):
        original_value = x[index]

        x[index] = original_value + h
        value_right = f(x)

        x[index] = original_value - h
        value_left = f(x)

        x[index] = original_value

        gradient[index] = (value_right - value_left) / (2 * h)

    return gradient


def relative_error(analytical: np.ndarray, numerical: np.ndarray) -> float:
    """Return a scale-aware maximum difference between two gradients."""
    numerator = np.abs(analytical - numerical)
    denominator = np.maximum(1e-8, np.abs(analytical) + np.abs(numerical))
    return float(np.max(numerator / denominator))


x = np.array([1.5, -2.0, 0.5], dtype=np.float64)

# f(x) = sum(x^3), so df/dx = 3x^2.
numerical = eval_numerical_gradient(lambda value: np.sum(value**3), x)
analytical = 3 * x**2

print("analytical:", analytical)
print("numerical: ", numerical)
print("relative error:", relative_error(analytical, numerical))
assert relative_error(analytical, numerical) < 1e-8


analytical: [ 6.75 12.    0.75]
numerical:  [ 6.75 12.    0.75]
relative error: 4.028296214953492e-11


### Interpreting gradient-check results

A tiny relative error supports the analytical implementation, but it is not a mathematical proof. A large error can come from:

- an incorrect derivative;
- a missing branch contribution;
- an incorrect upstream gradient;
- a shape or broadcasting mistake;
- in-place mutation of cached values;
- a poorly chosen $h$;
- a nondifferentiable point such as ReLU at zero;
- low-precision inputs.

For nondifferentiable functions, the centered numerical slope may disagree with the implementation's chosen convention even when the backward code is intentional.


## 8. A complete affine–ReLU graph

Consider the scalar loss

$$
s=xW+b,
$$

$$
h=\operatorname{ReLU}(s),
$$

$$
L=\frac{1}{2}\sum_j h_j^2.
$$

The factor $1/2$ makes the first backward step simple:

$$
\frac{\partial L}{\partial h}=h.
$$

Then backpropagate in reverse order:

1. squared loss gives $dh=h$;
2. ReLU gives $ds=dh\odot\mathbb{1}[s>0]$;
3. affine backward gives $dx$, $dW$, and $db$.

Notice that the forward order is affine → ReLU → loss, while the backward order is loss → ReLU → affine.


### Exercise 7 — Implement and check the complete graph

Use your affine and ReLU functions. Then numerically check the gradient with respect to `W`. The chosen values keep pre-activations away from zero so the graph is differentiable around the check point.


In [44]:
x = np.array([1.0, -2.0], dtype=np.float64)
W = np.array([[0.5, -1.0, 2.0], [-0.5, 0.25, 1.0]], dtype=np.float64)
b = np.array([0.1, 0.2, -0.3], dtype=np.float64)

# Forward: affine -> ReLU -> squared loss.
scores, affine_cache = affine_forward_single(x, W, b)
hidden, relu_cache = relu_forward(scores)
loss = 1/2 * np.sum(hidden*hidden)

# Backward: squared loss -> ReLU -> affine.
dhidden = 1 * hidden
dscores = relu_backward(dhidden, relu_cache)
dx, dW, db = affine_backward_single(dscores, affine_cache)

def loss_from_W(candidate_W: np.ndarray) -> float:
    """Recompute the same scalar loss for numerical differentiation."""
    s = x @ candidate_W + b
    h = np.maximum(0.0, s)
    return 1/2 * np.sum(h*h)


numerical_dW = eval_numerical_gradient(loss_from_W, W)
error = relative_error(dW, numerical_dW)

print("scores:", scores)
print("hidden:", hidden)
print("loss:", loss)
print("analytical dW:\n", dW)
print("numerical dW:\n", numerical_dW)
print("relative error:", error)

assert np.isscalar(loss)
assert dx.shape == x.shape
assert dW.shape == W.shape
assert db.shape == b.shape
assert error < 1e-8


scores: [ 1.6 -1.3 -0.3]
hidden: [1.6 0.  0. ]
loss: 1.2800000000000002
analytical dW:
 [[ 1.6  0.   0. ]
 [-3.2 -0.  -0. ]]
numerical dW:
 [[ 1.6  0.   0. ]
 [-3.2  0.   0. ]]
relative error: 3.969394257714374e-12


## 9. Backpropagation checklist

When deriving or debugging a backward pass:

1. Write every forward intermediate and its shape.
2. Start the scalar loss with gradient $1$.
3. Visit operations in reverse forward order.
4. At each operation, multiply the upstream gradient by its local derivative.
5. Add contributions when a value reaches the loss through multiple paths.
6. Confirm every returned gradient has the same shape as its corresponding input.
7. Avoid mutating upstream gradients or cached forward values unexpectedly.
8. Compare analytical and numerical gradients in `float64`.
9. Keep gradient-check inputs away from nondifferentiable points.
10. Test on tiny values that can also be calculated by hand.

## Reflection

Add a Markdown cell answering:

1. What information flows during the forward pass and what flows during the backward pass?
2. What is the difference between an upstream gradient and a local derivative?
3. Why does the backward pass begin with $\partial L/\partial L=1$?
4. When do we multiply gradient factors, and when do we add them?
5. Why did $x$ receive two gradient contributions in Exercise 2?
6. Why must a gradient have the same shape as the value it differentiates?
7. What role does the upstream vector play in affine backward?
8. Why is ReLU nondifferentiable at zero, and which convention did we use?
9. Why can the smallest numerical-gradient error occur at neither the largest nor the smallest $h$?
10. What can cause a numerical gradient check to fail even when a backward implementation is intentional?
11. Why is numerical differentiation suitable for checking but unsuitable for training?
12. In Exercise 7, why must the operations be visited in reverse order?

After this notebook, we will move the numerical-gradient helper into `cs231n_practice/gradient_check.py` with focused tests, then study batched affine and ReLU layers.


1. Forward pass - calculate output from input data. Backward pass - loss function derivatives propagation
2. Upstream gradient comes from next level(s) functions. Local derivative considers only this level function derivative
3. If we derive loss by loss it is 1 - it is its upstream derivative
4. We multiply upstream and local derivatives along one path using the chain rule. We add gradient contributions when the same value influences the loss through multiple branches.
5. cause both a and b depend on x
6. A gradient contains one partial derivative for every element of the corresponding value. Therefore, dW has the shape of W, db has the shape of b, and dx has the shape of x. Matching shapes also allows elementwise parameter updates.
7. ds represents $\partial L/\partial s$: how the final loss changes with every affine output. The affine backward pass combines ds with the cached inputs and weights to calculate dx, dW, and db. Without ds, the affine operation would know only its local derivatives, not its effect on the final loss.
8. zero is edge point and there is no clear slope there - it might 0 or 1 with the same reasoning. we used if x > 0 than derivative 1 and 0 otherwise
9. A large $h$ gives truncation error because the approximation is not sufficiently local. An extremely small $h$ causes floating-point cancellation and rounding error when subtracting nearly equal values. The best $h$ balances these two errors.
10. A check can disagree at a nondifferentiable point such as ReLU at zero. The implementation may intentionally choose derivative zero, while the centered numerical difference observes behavior from both sides. Poor $h$, low precision, and values extremely close to the boundary can also cause disagreement.
11. it is computational heavy
12. during backpropagation we move from loss function to input